In [ ]:
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


df = pd.read_csv("diabetes_dataset.csv")


X = df.drop("Outcome", axis=1)
y = df["Outcome"]

# %80 Eğitim, %20 Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

print("Eğitim seti boyutu:", X_train.shape)
print("Test seti boyutu:", X_test.shape)

Eğitim seti boyutu: (614, 8)
Test seti boyutu: (154, 8)


In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(16, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(8, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation="sigmoid"),
])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 16)             │           144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 289 (1.13 KB)

 Trainable params: 289 (1.13 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),
    loss=keras.losses.BinaryCrossentropy(),
    metrics=[keras.metrics.BinaryAccuracy(name="accuracy")]
)

early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=32,
    verbose=2,
    callbacks=[early_stopping]
)

Epoch 1/20
16/16 - 2s - 140ms/step - accuracy: 0.5723 - loss: 0.6799 - val_accuracy: 0.6098 - val_loss: 0.6599
Epoch 2/20
16/16 - 0s - 24ms/step - accuracy: 0.5988 - loss: 0.6665 - val_accuracy: 0.6098 - val_loss: 0.6574
Epoch 3/20
16/16 - 0s - 19ms/step - accuracy: 0.5703 - loss: 0.6648 - val_accuracy: 0.6016 - val_loss: 0.6551
Epoch 4/20
16/16 - 0s - 14ms/step - accuracy: 0.5621 - loss: 0.6798 - val_accuracy: 0.6016 - val_loss: 0.6529
Epoch 5/20
16/16 - 0s - 13ms/step - accuracy: 0.6191 - loss: 0.6574 - val_accuracy: 0.6098 - val_loss: 0.6505
Epoch 6/20
16/16 - 0s - 14ms/step - accuracy: 0.6008 - loss: 0.6633 - val_accuracy: 0.6098 - val_loss: 0.6481
Epoch 7/20
16/16 - 0s - 20ms/step - accuracy: 0.6130 - loss: 0.6697 - val_accuracy: 0.6098 - val_loss: 0.6461
Epoch 8/20
16/16 - 0s - 12ms/step - accuracy: 0.5845 - loss: 0.6687 - val_accuracy: 0.6098 - val_loss: 0.6442
Epoch 9/20
16/16 - 0s - 20ms/step - accuracy: 0.6008 - loss: 0.6642 - val_accuracy: 0.6179 - val_loss: 0.6422
Epoch 10/

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest kaybı (Loss): {test_loss:.4f} | Test doğruluğu (Accuracy): {test_acc:.4f}")


Test kaybı (Loss): 0.6192 | Test doğruluğu (Accuracy): 0.6494
